In [3]:
import openmc
import numpy as np 
print(f"OpenMC Version: {openmc.__version__}")
print("OpenMC imported successfully!")

OpenMC Version: 0.15.2
OpenMC imported successfully!


In [4]:
import openmc.data
print(openmc.config)

{'resolve_paths': True, 'cross_sections': PosixPath('/root/nndc_hdf5/cross_sections.xml')}


In [5]:
# Define UO2 fuel material (3.1% enriched)
uo2 = openmc.Material(name="UO2 Fuel")
uo2.set_density("g/cm3", 10.29)
uo2.add_nuclide("U235", 3.1,  percent_type="ao")   # 3.1% enriched
uo2.add_nuclide("U238", 96.9, percent_type="ao")   # remainder
uo2.add_nuclide("O16",  200.0, percent_type="ao")  # 2 oxygen per U

# Define water moderator
water = openmc.Material(name="Water")
water.set_density("g/cm3", 0.7)
water.add_nuclide("H1",  2.0, percent_type="ao")
water.add_nuclide("O16", 1.0, percent_type="ao")
water.add_s_alpha_beta("c_H_in_H2O")   # thermal scattering

# Collect materials
materials = openmc.Materials([uo2, water])
materials.export_to_xml()
print("Materials defined!")
print(f"UO2 density: {uo2.density} g/cm3")
print(f"Water density: {water.density} g/cm3")

Materials defined!
UO2 density: 10.29 g/cm3
Water density: 0.7 g/cm3


In [6]:
# Fuel pin radius
fuel_radius = openmc.ZCylinder(r=0.4096)

# Pin cell boundary (square)
pitch = 1.26  # cm
left   = openmc.XPlane(-pitch/2, boundary_type="reflective")
right  = openmc.XPlane(+pitch/2, boundary_type="reflective")
bottom = openmc.YPlane(-pitch/2, boundary_type="reflective")
top    = openmc.YPlane(+pitch/2, boundary_type="reflective")

# Define regions
fuel_region  = -fuel_radius
water_region = +fuel_radius & -right & +left & -top & +bottom

# Define cells
fuel_cell  = openmc.Cell(name="fuel",  fill=uo2,   region=fuel_region)
water_cell = openmc.Cell(name="water", fill=water, region=water_region)

# Define universe and geometry
universe  = openmc.Universe(cells=[fuel_cell, water_cell])
geometry  = openmc.Geometry(universe)
geometry.export_to_xml()

print("Geometry defined!")
print(f"Fuel radius : {fuel_radius.r} cm")
print(f"Cell pitch  : {pitch} cm")

Geometry defined!
Fuel radius : 0.4096 cm
Cell pitch  : 1.26 cm


In [9]:
# --- Settings ---
settings = openmc.Settings()
settings.batches         = 150      # total batches
settings.inactive        = 50       # skip first 50 (warmup)
settings.particles       = 10000    # neutrons per batch
settings.export_to_xml()

# --- Energy group boundaries (eV) ---
group_edges = np.array([
    1.0e-5,    # bottom of group 4 (thermal)
    6.25e-1,   # G3/G4 boundary
    5.53e3,    # G2/G3 boundary
    8.21e5,    # G1/G2 boundary
    2.0e7      # top of group 1 (fast)
])

# --- Tallies ---
tallies = openmc.Tallies()

# Flux tally per energy group
flux_tally = openmc.Tally(name="flux")
flux_tally.filters = [
    openmc.EnergyFilter(group_edges)
]
flux_tally.scores = ["flux"]
tallies.append(flux_tally)

# Reaction rate tallies
rxn_tally = openmc.Tally(name="reactions")
rxn_tally.filters = [
    openmc.EnergyFilter(group_edges)
]
rxn_tally.scores = [
    "flux",
    "fission",
    "absorption",
    "nu-fission",
    "scatter"
]
tallies.append(rxn_tally)
tallies.export_to_xml()
print("Tallies defined!")
print(f"Energy groups : {len(group_edges)-1}")
print(f"Group edges   : {group_edges}")

Tallies defined!
Energy groups : 4
Group edges   : [1.00e-05 6.25e-01 5.53e+03 8.21e+05 2.00e+07]


/openmc_venv/lib/python3.11/site-packages/openmc/mixin.py:70: IDWarning: Another Filter instance already exists with id=5.
  warn(msg, IDWarning)


In [11]:
openmc.run()
print("Simulation Complete")

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

In [12]:
import openmc
import numpy as np

# Load results
sp = openmc.StatePoint("statepoint.150.h5")

# Get reaction tally
rxn_tally = sp.get_tally(name="reactions")

# Extract mean values
flux       = rxn_tally.get_values(scores=["flux"])
fission    = rxn_tally.get_values(scores=["fission"])
absorption = rxn_tally.get_values(scores=["absorption"])
nu_fission = rxn_tally.get_values(scores=["nu-fission"])
scatter    = rxn_tally.get_values(scores=["scatter"])

# Reshape to (energy_groups,)
G = 4
flux       = flux.reshape(G)
fission    = fission.reshape(G)
absorption = absorption.reshape(G)
nu_fission = nu_fission.reshape(G)
scatter    = scatter.reshape(G)

print("Raw tally results (per group):")
print(f"{'Group':<8}{'Flux':<15}{'Fission':<15}{'Absorption':<15}")
for g in range(G):
    print(f"  G{g+1:<6}{flux[g]:<15.4e}{fission[g]:<15.4e}{absorption[g]:<15.4e}")

Raw tally results (per group):
Group   Flux           Fission        Absorption     
  G1     6.2821e+00     4.6600e-01     6.6318e-01     
  G2     1.0967e+01     6.2047e-02     2.6637e-01     
  G3     1.3306e+01     5.2413e-03     3.0843e-02     
  G4     1.0443e+01     3.4173e-02     4.2006e-02     


In [15]:
# Compute group cross sections
sigma_f    = fission    / flux
sigma_a    = absorption / flux
nu_sigma_f = nu_fission / flux
sigma_s    = scatter    / flux          # ← add this!
sigma_t    = sigma_a + sigma_s          # ← total XS
D          = 1.0 / (3.0 * sigma_t)     # ← correct D!

print("\n4-Group Cross Sections (volume-averaged):")
print(f"{'Group':<8}{'D':<10}{'sigma_a':<12}{'sigma_s':<12}{'nu_sigma_f':<14}")
for g in range(G):
    print(f"  G{g+1:<6}"
          f"{D[g]:<10.4f}"
          f"{sigma_a[g]:<12.6f}"
          f"{sigma_s[g]:<12.6f}"
          f"{nu_sigma_f[g]:<14.6f}")


4-Group Cross Sections (volume-averaged):
Group   D         sigma_a     sigma_s     nu_sigma_f    
  G1     0.2159    0.105566    1.438599    0.180752      
  G2     0.3773    0.024287    0.859239    0.013780      
  G3     0.6024    0.002318    0.551013    0.000966      
  G4     1.5219    0.004022    0.215000    0.009130      


In [17]:
import os
import glob

# Delete previous OpenMC output files
for f in glob.glob("statepoint.*.h5"):
    os.remove(f)
    print(f"Deleted: {f}")

for f in ["summary.h5", "tallies.out"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted: {f}")

print("Clean! Ready to rerun.")

Deleted: statepoint.150.h5
Deleted: summary.h5
Deleted: tallies.out
Clean! Ready to rerun.


In [18]:
# Rerun with cell filter to get material-specific XS
tallies2 = openmc.Tallies()

# Cell filter separates fuel from water
cell_filter = openmc.CellFilter([fuel_cell, water_cell])

rxn_tally2 = openmc.Tally(name="reactions_by_cell")
rxn_tally2.filters = [
    openmc.EnergyFilter(group_edges),
    cell_filter                          # ← separates by cell!
]
rxn_tally2.scores = [
    "flux", "fission",
    "absorption", "nu-fission", "scatter"
]
tallies2.append(rxn_tally2)
tallies2.export_to_xml()

# Rerun simulation
openmc.run()
print("Second simulation complete!")

                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

In [19]:
# Load new statepoint
sp2 = openmc.StatePoint("statepoint.150.h5")
tally2 = sp2.get_tally(name="reactions_by_cell")

# Extract for each cell separately
results = {}
for cell_name, cell_obj in [("fuel", fuel_cell), 
                              ("water", water_cell)]:
    
    # Filter by cell
    cell_tally = tally2.get_slice(
        filters=[openmc.CellFilter],
        filter_bins=[(cell_obj.id,)]
    )
    
    flux_c  = cell_tally.get_values(
                scores=["flux"]).reshape(G)
    abs_c   = cell_tally.get_values(
                scores=["absorption"]).reshape(G)
    nuf_c   = cell_tally.get_values(
                scores=["nu-fission"]).reshape(G)
    scat_c  = cell_tally.get_values(
                scores=["scatter"]).reshape(G)
    
    # Compute XS
    sigma_a  = abs_c  / flux_c
    nu_sig_f = nuf_c  / flux_c
    sigma_s  = scat_c / flux_c
    sigma_t  = sigma_a + sigma_s
    D        = 1.0 / (3.0 * sigma_t)
    
    results[cell_name] = {
        "D"          : D.tolist(),
        "sigma_a"    : sigma_a.tolist(),
        "nu_sigma_f" : nu_sig_f.tolist(),
        "sigma_s"    : sigma_s.tolist(),
    }
    
    print(f"\n{cell_name.upper()} Cross Sections:")
    print(f"{'G':<4}{'D':<10}{'sigma_a':<12}{'nu_sigma_f':<14}")
    for g in range(G):
        print(f"  {g+1:<3}"
              f"{D[g]:<10.4f}"
              f"{sigma_a[g]:<12.6f}"
              f"{nu_sig_f[g]:<14.6f}")


FUEL Cross Sections:
G   D         sigma_a     nu_sigma_f    
  1  0.4625    0.324166    0.595353      
  2  0.6222    0.073606    0.042492      
  3  0.7760    0.006820    0.002850      
  4  1.2221    0.011188    0.026453      

WATER Cross Sections:
G   D         sigma_a     nu_sigma_f    
  1  0.1751    0.010264    0.000000      
  2  0.3173    0.000618    0.000000      
  3  0.5404    0.000009    0.000000      
  4  1.7479    0.000246    0.000000      


In [20]:
import yaml

# Build scattering matrix (simplified downscatter only)
# We'll use sigma_s from volume-averaged calculation
def build_scatter_matrix(sigma_s_total, G):
    """
    Build G×G scattering matrix.
    Simplified: all scatter goes to next group only.
    """
    S = [[0.0]*G for _ in range(G)]
    for g in range(G-1):
        S[g][g+1] = float(sigma_s_total[g])
    return S

# Build ANGKOR-compatible materials dict
angkor_materials = {}

for mat_name, xs in results.items():
    sigma_s_total = xs["sigma_s"]
    S = build_scatter_matrix(sigma_s_total, G)

    angkor_materials[mat_name] = {
        "groups"     : G,
        "D"          : [round(x, 6) for x in xs["D"]],
        "sigma_a"    : [round(x, 6) for x in xs["sigma_a"]],
        "nu_sigma_f" : [round(x, 6) for x in xs["nu_sigma_f"]],
        "chi"        : [1.0, 0.0, 0.0, 0.0],
        "sigma_s"    : S,
    }

# Save to YAML
output_path = "/angkor/input/pwr_xs_openmc.yaml"
with open(output_path, "w") as f:
    yaml.dump(angkor_materials, f,
              default_flow_style=False,
              sort_keys=False)

print(f"Saved: {output_path}")
print("\nFinal cross sections for ANGKOR:")
for mat, xs in angkor_materials.items():
    print(f"\n{mat}:")
    print(f"  D         = {xs['D']}")
    print(f"  sigma_a   = {xs['sigma_a']}")
    print(f"  nu_sigma_f= {xs['nu_sigma_f']}")

Saved: /angkor/input/pwr_xs_openmc.yaml

Final cross sections for ANGKOR:

fuel:
  D         = [0.462521, 0.622224, 0.776047, 1.222129]
  sigma_a   = [0.324166, 0.073606, 0.00682, 0.011188]
  nu_sigma_f= [0.595353, 0.042492, 0.00285, 0.026453]

water:
  D         = [0.175146, 0.317324, 0.540404, 1.747897]
  sigma_a   = [0.010264, 0.000618, 9e-06, 0.000246]
  nu_sigma_f= [0.0, 0.0, 0.0, 0.0]


In [21]:
print(0.011188 + 1.222129 * 6.9e-4)

0.01203126901


In [23]:
B2 = 6.9e-4

corrected_materials = {}
for mat_name, xs in results.items():
    sigma_a_corr = [
        xs["sigma_a"][g] + xs["D"][g] * B2
        for g in range(G)
    ]
    corrected_materials[mat_name] = dict(xs)
    corrected_materials[mat_name]["sigma_a"] = sigma_a_corr

print("B² correction applied to ALL materials!")
for mat in corrected_materials:
    print(f"\n{mat} sigma_a (corrected):")
    print(f"  {corrected_materials[mat]['sigma_a']}")

B² correction applied to ALL materials!

fuel sigma_a (corrected):
  [0.32448547291567087, 0.0740356152791365, 0.007355803724542281, 0.012030813284308443]

water sigma_a (corrected):
  [0.010384801327002618, 0.0008369392151585917, 0.00038189169602271834, 0.0014518033883017191]


In [24]:
import yaml

angkor_corrected = {}
for mat_name, xs in corrected_materials.items():
    S = build_scatter_matrix(xs["sigma_s"], G)
    angkor_corrected[mat_name] = {
        "groups"     : G,
        "D"          : [round(x,6) for x in xs["D"]],
        "sigma_a"    : [round(x,6) for x in xs["sigma_a"]],
        "nu_sigma_f" : [round(x,6) for x in xs["nu_sigma_f"]],
        "chi"        : [1.0, 0.0, 0.0, 0.0],
        "sigma_s"    : S,
    }

with open("/angkor/input/pwr_xs_corrected.yaml", "w") as f:
    yaml.dump(angkor_corrected, f,
              default_flow_style=False,
              sort_keys=False)
print("Saved: pwr_xs_corrected.yaml")

Saved: pwr_xs_corrected.yaml
